1. Creacion de corte para pacientes con WSI y mRNA

In [ ]:
#Librerias
from __future__ import annotations
import re
from pathlib import Path
from typing import Optional
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

#Hiperparametros
RANDOM_STATE = 42
TEST_SIZE = 0.15
VAL_SIZE = 0.15

#Rutas
MRNA_PATH = Path(r"D:\ProyectoPDAC\Omics\mRNA_RSEM_UQ_log2_Tumor.cct") #ruta de abundancias relativas de mRNA
PATCH_ROOT = Path(r"D:\ProyectoPDAC\Parches_PDAC_Macenko\tumor") #ruta parches
TCIA_METADATA_CSV = Path(r"C:\Users\Usuario\Python\Tesis\TCIA CPTAC Pathology Portal.csv") #ruta de informacion general

OUT_DIR = Path(r"C:\Users\Usuario\Python\Tesis\outputs_pipeline_wsi_omics\mRNA_UNI_MIL_1") #ruta salida
OUT_DIR.mkdir(parents=True, exist_ok=True)

#Funciones
def extract_case_id(text: str) -> Optional[str]: #extraccion del id a partir de str o texto
    text = str(text)
    match = re.search(r"(C3[LN]-\d{5})", text, flags=re.IGNORECASE) #identifica el id en base al formato establecido
    return match.group(1).upper() if match else None #regresa el id en mayusculas


def extract_slide_id_from_patch_name(filename: str) -> Optional[str]: #extraccion de id a partir del nombre del parche
    name = Path(filename).stem
    match = re.search(r"(C3[LN]-\d{5}-\d+)", name, flags=re.IGNORECASE) #identifica el id en base al formato establecido
    return match.group(1).upper() if match else None #regresa el id en mayusculas


def extract_xy_from_patch_name(filename: str): #extraccion de coordenadas del parche a partir del nombre del parche
    name = Path(filename).stem

    mx = re.search(r"_x(\d+)", name, flags=re.IGNORECASE) #busca la coordenada x basado en el nombre del archivo
    my = re.search(r"_y(\d+)", name, flags=re.IGNORECASE) #busca la coordenada y basado en el nombre del archivo
    x = int(mx.group(1)) if mx else None  #convierte en entero y si no existe regresa None
    y = int(my.group(1)) if my else None

    return x, y #regresa las coordenadas obtenidas



#Carga la matriz mRNA
print("Lectura de la matriz mRNA")

mrna_raw = pd.read_csv(MRNA_PATH, sep="\t", low_memory=False) #lee el archivo por completo

print("Forma original mRNA:", mrna_raw.shape) #mustra el tamaño de la matriz
print("Primeras columnas:", mrna_raw.columns[:10].tolist()) #muestra las primeras columnas para revisar

gene_col = mrna_raw.columns[0] #guardamos la columna correspondiente a los genes
mrna_raw = mrna_raw.rename(columns={gene_col: "Gene"}) #la renombramos
mrna_raw["Gene"] = mrna_raw["Gene"].astype(str) #lo identifiacamos como texto

patient_cols = [] #lista para guardar los pacientes
patient_map = {} #diccionario para relacionar cada nombre de paciente con el case id

for col in mrna_raw.columns: #bucle que recorre todas las columnas de la matriz mRNA
    case_id = extract_case_id(col)
    if case_id is not None:
        patient_cols.append(col)
        patient_map[col] = case_id

print("Columnas de pacientes detectadas en mRNA:", len(patient_cols)) #mustra cuantas columnas se encontraron

if len(patient_cols) == 0:
    raise ValueError("No se detectaron pacientes en la matriz mRNA") #si no se encuentran se detiene

mrna = mrna_raw[["Gene"] + patient_cols].copy() #crea una copia que conserva solo columans de genes y pacientes
mrna = mrna.rename(columns=patient_map) #renombra las columnas con case id

mrna_numeric = mrna.drop(columns=["Gene"]).apply(pd.to_numeric, errors="coerce") #convierte a valores numericos la columna de expresion genica
mrna_numeric["Gene"] = mrna["Gene"] #se añade la coluna de genes con valores numericos

#genes x pacientes
mrna_by_gene = mrna_numeric.groupby("Gene").mean()   #agrupa por gen y calcula el promedio

#pacientes x genes
mrna_patient = mrna_by_gene.T  #aplicamos la matriz traspuesta
mrna_patient.index.name = "Case_ID" #agregamos el case id al indice (cada fila un paciente)
mrna_patient = mrna_patient.reset_index() #convierte el indice en una columna normal del dataframe

print("Matriz mRNA paciente x genes:", mrna_patient.shape) #muestra la forma final de la matriz


#Indexa parches tumorales
print("Indexando patches tumorales")

if not PATCH_ROOT.exists(): #bucle que verifica que exista la carpeta de los parches tumor
    raise FileNotFoundError(f"No existe PATCH_ROOT: {PATCH_ROOT}")

records = [] #lista de almacenamiento de informacion de los parches

patient_dirs = sorted([p for p in PATCH_ROOT.iterdir() if p.is_dir()]) #conserva elementos que sean carpetas

for patient_dir in patient_dirs: #bucle que recorre todas las carpetas #busca las carpetas de los pacientes dentro de patch root
    case_id = extract_case_id(patient_dir.name) #extrae el id del paciente

    if case_id is None: #omite si no hay
        continue

    patch_files = sorted(patient_dir.glob("*.png")) #busca todos los archivos .png dentro de la carpeta del paciente

    for patch_path in patch_files:    #recorre todos los parches del paciente
        slide_id = extract_slide_id_from_patch_name(patch_path.name) #extrae el id 
        x, y = extract_xy_from_patch_name(patch_path.name) #extrae coordenadas

        if slide_id is None: #omite si no hay
            continue

        records.append(     #se guarda la informacion del parche en forma de deiccionario
            {
                "Case_ID": case_id,
                "Slide_ID": slide_id,
                "patch_name": patch_path.name,
                "patch_path": str(patch_path),
                "x": x,
                "y": y,
            }
        )

patch_index = pd.DataFrame(records) #tranforma la lista de diccionarios a una tabla pandas

if patch_index.empty:
    raise ValueError("No se encontraron parches tumorales") #se detiene si no hay parches

print("Patches encontrados:", len(patch_index)) #muestra el total de parches indexados
print("Pacientes con patches:", patch_index["Case_ID"].nunique()) #muestra cuantos pacientes tienen parches disponibles
print("Slides detectados:", patch_index["Slide_ID"].nunique()) #muestra el numero de slides que se detectaron


#Resumen estadistico a nivel slide
print("Construyendo resumen por slide")

slide_summary = (                     #agrupa todos los parches que pertenecen a la misma slide
    patch_index.groupby(["Case_ID", "Slide_ID"]) #agrupa por paciente y por slide
    .agg(                                        #calculo de estadisticas descriptivas
        n_patches=("patch_path", "count"),      
        x_min=("x", "min"),
        x_max=("x", "max"),
        y_min=("y", "min"),
        y_max=("y", "max"),
        x_n_unique=("x", pd.Series.nunique),
        y_n_unique=("y", pd.Series.nunique),
    )
    .reset_index()  #tranformamos a columnas normales
    .sort_values(["Case_ID", "n_patches"], ascending=[True, False]) #ordena las slides por paciente y numero de parches
    .reset_index(drop=True) #reinicia la numeracion de filas
)

print("Slides resumidos:", len(slide_summary)) #mustra el total de slides resumidas


#Seleccion de una sola slide por paciente (la que tenga mas parches)
print("Seleccionando 1 slide por paciente")

selected_slides = (
    slide_summary   #usa el resumen generado anteriormente
    .sort_values(["Case_ID", "n_patches"], ascending=[True, False]) #ordena las slides de cada paciente en referencia al numero de parches
    .drop_duplicates(subset=["Case_ID"], keep="first") #los que tiene mas slides van primero. por lo que conserva la primera slide
    .rename(   #renombra algnas columnas con respecto a esta seleccion
        columns={
            "Slide_ID": "selected_slide_id",
            "n_patches": "selected_slide_n_patches",
        }
    )
    .reset_index(drop=True) #reinicia la numeracion de filas
)

print("Pacientes con slide seleccionada:", len(selected_slides)) #muestra el total de parches tras la seleccion
print("Resumen de parches por slide seleccionada:")            
print(selected_slides["selected_slide_n_patches"].describe()) #muestra las estadisticas


#Cruce de pacientes con datos de mRNA y WSI

print("Creando cohorte comun mRNA + WSI")

mrna_patients = set(mrna_patient["Case_ID"]) #convierte la columna de pacientes en un conjunto
wsi_patients = set(selected_slides["Case_ID"]) #convierte la columna de slides de pacientes en un conjunto

common_patients = sorted(mrna_patients.intersection(wsi_patients)) #identifica los pacientes repetidos en ambos conjuntos

print("Pacientes con mRNA:", len(mrna_patients)) #muetra cuantos pacientes hay en cada conjunto de datos
print("Pacientes con WSI seleccionada:", len(wsi_patients)) 
print("Pacientes comunes:", len(common_patients)) #muestra el tamaño total de pacientes con datos mRNA y WSI

cohort = selected_slides[selected_slides["Case_ID"].isin(common_patients)].copy()  #filtra la tabla de WSI con datos en comun
cohort = cohort.sort_values("Case_ID").reset_index(drop=True) #ordenamos por case id

mrna_common = mrna_patient[mrna_patient["Case_ID"].isin(common_patients)].copy() #filtra la matriz mRNA y conserva los pacientes en comun
mrna_common = mrna_common.sort_values("Case_ID").reset_index(drop=True) # ordena los pacientes

if len(cohort) != len(mrna_common): #verifica que ambas rablas tengan el mismo numero de pacientes en comun
    raise ValueError("No coincide el numero de pacientes entre WSI y mRNA")


#Split por paciente
print("Realizando split por paciente")

all_patients = cohort["Case_ID"].tolist() #carga la lista final de pacientes disponibles

trainval_patients, test_patients = train_test_split( #separamos en 2 grupos (train+val y test)
    all_patients,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    shuffle=True
)

relative_val_size = VAL_SIZE / (1 - TEST_SIZE) #calcula el tamaño relativo respecto a train + val

train_patients, val_patients = train_test_split( #separamos el train+val (train y val)
    trainval_patients,
    test_size=relative_val_size,
    random_state=RANDOM_STATE,
    shuffle=True
)

cohort["split"] = "unassigned" #crea una columna de asignacion para el split
cohort.loc[cohort["Case_ID"].isin(train_patients), "split"] = "train" #marca a los pacientes como train
cohort.loc[cohort["Case_ID"].isin(val_patients), "split"] = "val" #marca a los pacientes como val
cohort.loc[cohort["Case_ID"].isin(test_patients), "split"] = "test" #marca a los pacientes como test

print("\nDistribución por split:")
print(cohort["split"].value_counts()) #mustra cuantos pacientes hay por grupo

print("\nParches de slide seleccionada por split:") 
print(cohort.groupby("split")["selected_slide_n_patches"].sum()) #mustra el numero de parches totales de cada grupo


# Control leakage
sets = {                           #convierte las listas de pacientes en conjuntos
    "train": set(train_patients),
    "val": set(val_patients),
    "test": set(test_patients),
}

assert sets["train"].isdisjoint(sets["val"])  #comparamos intersecciones entre pacientes de cada grupo
assert sets["train"].isdisjoint(sets["test"])
assert sets["val"].isdisjoint(sets["test"])

print("Sin leakage entre splits")  #si todo es verdadero se muestra el mensaje


#Guarda las salidas

print("Guardando archivos")

patch_index_path = OUT_DIR / "mRNA_UNI_MIL_1_patch_index_tumor.csv" #indice de parches
slide_summary_path = OUT_DIR / "mRNA_UNI_MIL_1_slide_summary_tumor.csv" #resumen estadistico por slide
selected_slides_path = OUT_DIR / "mRNA_UNI_MIL_1_selected_slide_per_patient.csv" #slide seleccionada por paciente
cohort_split_path = OUT_DIR / "mRNA_UNI_MIL_1_patient_split_selected_slide.csv" #split de pacientes (train, val, test)
mrna_common_path = OUT_DIR / "mRNA_UNI_MIL_1_mRNA_common_patients.csv" #matris de pacientes comunes (simultaneos en las dos fuentes de datos)
summary_path = OUT_DIR / "mRNA_UNI_MIL_1_summary.txt" #resumen del procesamiento

patch_index.to_csv(patch_index_path, index=False) #guarda el indice de todos los parches
slide_summary.to_csv(slide_summary_path, index=False) #guarda el resumen por slide
selected_slides.to_csv(selected_slides_path, index=False) #guarda la slide seleccionada por paciente
cohort.to_csv(cohort_split_path, index=False) #guarda la corte final con los splits
mrna_common.to_csv(mrna_common_path, index=False) #guarda la matriz mRNA filtrada

with open(summary_path, "w", encoding="utf-8") as f: #archivo txt para resumen
    f.write("mRNA_UNI_MIL_1\n")
    f.write("Cohorte común mRNA + WSI tumor con 1 slide seleccionada por paciente\n\n")
    f.write(f"Pacientes mRNA: {len(mrna_patients)}\n") #anota el total de pacientes con mRNA
    f.write(f"Pacientes con patches tumorales: {len(wsi_patients)}\n") #numero total de pacientes con WSI
    f.write(f"Pacientes comunes: {len(common_patients)}\n\n") #pacientes en comun
    f.write("Split:\n")
    f.write(str(cohort["split"].value_counts())) #anota la cantidad de pacientes en cada split
    f.write("\n\nParches por split:\n")
    f.write(str(cohort.groupby("split")["selected_slide_n_patches"].sum())) #suma de parches por cada grupo
    f.write("\n\nControl de leakage: OK\n") #anota la confirmacion de no leakage

print("\nArchivos generados:") #muestra los archivos generados
print(patch_index_path)
print(slide_summary_path)
print(selected_slides_path)
print(cohort_split_path)
print(mrna_common_path)
print(summary_path)


In [ ]:
#Verificacion de Genes semilla

#Libreria
import pandas as pd


#cargamos el indice de pacientes generado en la etapa anterior
mrna_path = r"C:\Users\Usuario\Python\Tesis\outputs_pipeline_wsi_omics\mRNA_UNI_MIL_1\mRNA_UNI_MIL_1_mRNA_common_patients.csv"

df = pd.read_csv(mrna_path) #lee la matriz mRNA

print("Forma del dataset:", df.shape) #mustra la dimencion del dataset

#Obtiene la lista de genes del dataset
genes_dataset = set(df.columns)
genes_dataset.discard("Case_ID")  #quita columna de pacientes

#convierte en mayusculas los nombres de los genes
genes_dataset = set([g.upper() for g in genes_dataset]) 

print("Total genes disponibles:", len(genes_dataset)) #mustra el numero total de genes

#Genes semilla
basal_genes = ["TP63", "KRT5", "KRT6A", "KRT14", "KRT17", "LAMC2", "S100A2", "LOX"] #de tipo basal

classical_genes = ["PDX1", "GATA6", "HNF1A", "HNF4A", "FOXA2", "FOXA3", "MNX1", "HNF1B"] #de tipo clasical


# Funcion para verificar la presencia de gnees en la matriz
def check_genes(gene_list, genes_dataset):
    present = [] #genes encontrados
    missing = [] #no encontrados
    
    for g in gene_list:  #bucle que recorre la lista de genes
        if g.upper() in genes_dataset: #verifica si el gen existe
            present.append(g) #guarda el gen encontrado
        else:
            missing.append(g) #guarda el gen faltante
    
    return present, missing #regresa los genes encontrados y faltantes


#Verifica genes basal
basal_present, basal_missing = check_genes(basal_genes, genes_dataset)

#muestra
print("\nBasal")
print("Presentes:", basal_present)
print("Faltantes:", basal_missing)

#Verifica genes classical
classical_present, classical_missing = check_genes(classical_genes, genes_dataset)

#muestra
print("\nCLASSICAL")
print("Presentes:", classical_present)
print("Faltantes:", classical_missing)

#Muestra el resumen final
print("\nRESUMEN")
print(f"Basal: {len(basal_present)}/{len(basal_genes)} presentes")
print(f"Classical: {len(classical_present)}/{len(classical_genes)} presentes")

2. Creacion de firma molecular Basal y Clasical

In [ ]:
#Firma molecular basal vs classical (solo con el grupo train)

#Librerias
import pandas as pd
import numpy as np
from pathlib import Path

#Rutas
BASE_DIR = Path(r"C:\Users\Usuario\Python\Tesis\outputs_pipeline_wsi_omics\mRNA_UNI_MIL_1")

MRNA_PATH = BASE_DIR / "mRNA_UNI_MIL_1_mRNA_common_patients.csv"
SPLIT_PATH = BASE_DIR / "mRNA_UNI_MIL_1_patient_split_selected_slide.csv"

OUT_DIR = Path(r"C:\Users\Usuario\Python\Tesis\outputs_pipeline_wsi_omics\mRNA_UNI_MIL_2")
OUT_DIR.mkdir(parents=True, exist_ok=True)

#Parametros
VAR_THRESHOLD = 0.01 #varianza minima permitida
MAX_NAN_RATIO = 0.2  #porcentaje maximo de valores faltantes

#Semillas
basal_genes = ["TP63","KRT5","KRT6A","KRT14","KRT17","LAMC2","S100A2","LOX"]
classical_genes = ["PDX1","GATA6","HNF1A","HNF4A","FOXA2","FOXA3","MNX1","HNF1B"]

#Carga de datos
print("\nCargando datos")

mrna = pd.read_csv(MRNA_PATH) #carga la matriz transcriptomica
split_df = pd.read_csv(SPLIT_PATH) #carga la informacion de splits

print("mRNA shape:", mrna.shape) #muestra las dimenciones de la matriz mRNA
print("Split shape:", split_df.shape) #muestra las dimenciones de la tabla de splits

#separa el grupo train

print("Separando TRAIN")

train_ids = split_df[split_df["split"] == "train"]["Case_ID"] #filtra la columna split y extrae solo los ids que pertenecen a train

mrna_train = mrna[mrna["Case_ID"].isin(train_ids)].copy() #filtra la matriz de expresion genica y extrae solo las filas de los ids de train

print("Pacientes TRAIN:", len(mrna_train)) #mustra el total de pacientes asignados a entrenamiento

#Preprocesamiento

print("\nPreprocesamiento")

X_train = mrna_train.drop(columns=["Case_ID"]).copy() #elimina la columna de case id

#elimina genes con muchos NaNs
nan_ratio = X_train.isna().mean() #matriz booleana para calcular la proporcion de nulos en la matriz
valid_genes = nan_ratio[nan_ratio < MAX_NAN_RATIO].index #filtra los genes que cumplen con el criterio (pocos datos faltantes)

X_train = X_train[valid_genes] #filtra la matriz solo con los genes validos

print("Genes tras filtro NaN:", X_train.shape[1]) #mustra cuantos genes quedaron despues del filtro

#imputación (mediana)
X_train = X_train.fillna(X_train.median()) #reemplaza los valores faltantes con la mediana

#filtra por varianza
var = X_train.var() #calcula la varianza de cada gen
var_genes = var[var > VAR_THRESHOLD].index #conserva genes con variabilidad

X_train = X_train[var_genes] #filtra la matriz usando los genes con mayor varianza

print("Genes tras filtro varianza:", X_train.shape[1])

#Normalizacion de genes (z-score solo train)

print("\nNormalizando (Z-score)")

mean_train = X_train.mean() #calcula la media
std_train = X_train.std() #calcula la desviacion estandar de cada gen

X_train_z = (X_train - mean_train) / (std_train + 1e-8) #normaliza cada gen

#Verifica semillas

print("\nVerificando genes semilla")

genes_available = set(X_train_z.columns) #obtine los genes tras el el filtro anterior

basal_used = [g for g in basal_genes if g in genes_available] #selecciona los genes basal
classical_used = [g for g in classical_genes if g in genes_available] #selecciona los genes clasical

print("Basal usados:", basal_used)         #mustra los genes
print("Classical usados:", classical_used)

#Calcula scores (train)

print("\nCalculando scores")

score_basal = X_train_z[basal_used].mean(axis=1) #calcula score basal promedio por paciente
score_classical = X_train_z[classical_used].mean(axis=1) # calcula score classical promedio por paciente

labels = np.where(score_basal > score_classical, "basal", "classical") #asigna la etiqueta al score mas dominante

train_results = pd.DataFrame({                   #organiza los resultados una tabla para cada paciente
    "Case_ID": mrna_train["Case_ID"].values,     
    "score_basal": score_basal,
    "score_classical": score_classical,
    "label": labels
})

print(train_results["label"].value_counts()) #mustra los resultados de la tabla

#Aplica la firma molecular a toto el dataset (sin entrenar)

print("\nAplicando a todo el dataset")

X_all = mrna.drop(columns=["Case_ID"]) #elimina case id, queda solo expresion de genes

#Usa solo genes filtrados en train
X_all = X_all[var_genes]

#reemplaza NaN con mediana de train
X_all = X_all.fillna(mean_train)

#Normaliza con stats de train
X_all_z = (X_all - mean_train) / (std_train + 1e-8)

score_basal_all = X_all_z[basal_used].mean(axis=1) #calcula score basal para todos los pacientes
score_classical_all = X_all_z[classical_used].mean(axis=1) #calcula score classical para todos los pacientes

labels_all = np.where(score_basal_all > score_classical_all, "basal", "classical") # asigna la etiqueta molecular final

results_all = pd.DataFrame({                   #organiza los resultados una tabla para cada paciente
    "Case_ID": mrna["Case_ID"],
    "score_basal": score_basal_all,
    "score_classical": score_classical_all,
    "label": labels_all
})

#Guarda

train_path = OUT_DIR / "mRNA_UNI_MIL_2_train_labels.csv"
all_path = OUT_DIR / "mRNA_UNI_MIL_2_all_labels.csv"

train_results.to_csv(train_path, index=False) #guardalos resultados de train
results_all.to_csv(all_path, index=False) #guarda resultados de toda la cohorte

print("\nArchivos generados:")
print(train_path)
print(all_path)


3. Expancion de la firma molecular y refinamiento

In [ ]:
#Librerias
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import ttest_ind #prueba estadistica

#Rutas
BASE1 = Path(r"C:\Users\Usuario\Python\Tesis\outputs_pipeline_wsi_omics\mRNA_UNI_MIL_1")
BASE2 = Path(r"C:\Users\Usuario\Python\Tesis\outputs_pipeline_wsi_omics\mRNA_UNI_MIL_2")

MRNA_PATH = BASE1 / "mRNA_UNI_MIL_1_mRNA_common_patients.csv"
SPLIT_PATH = BASE1 / "mRNA_UNI_MIL_1_patient_split_selected_slide.csv"
INITIAL_LABELS_PATH = BASE2 / "mRNA_UNI_MIL_2_train_labels.csv"

OUT_DIR = Path(r"C:\Users\Usuario\Python\Tesis\outputs_pipeline_wsi_omics\mRNA_UNI_MIL_3")
OUT_DIR.mkdir(parents=True, exist_ok=True)

#parametros
MAX_NAN_RATIO = 0.20 #porcentaje maximo de valores faltantes permitido por gen
VAR_THRESHOLD = 0.01 #varianza minima requerida para conservar un gen

CORR_THRESHOLD = 0.35 #correlacion minima con la direccion basal-classical
FDR_THRESHOLD = 0.10 #umbral maximo de FDR para considerar un gen significativo
TOP_N_PER_CLASS = 50 #numero maximo de genes candidatos agregados a cada firma

#semillas
basal_seed = ["TP63","KRT5","KRT6A","KRT14","KRT17","LAMC2","S100A2","LOX"]
classical_seed = ["PDX1","GATA6","HNF1A","HNF4A","FOXA2","FOXA3","MNX1","HNF1B"]

seed_all = basal_seed + classical_seed

#Funcion FDR Benjamini-Hochberg (Para evitar el riesgo de falsos positivos por multiples comparacines)
def benjamini_hochberg(pvals):
    pvals = np.asarray(pvals) #convierte los p-values a un arreglo de numpy
    n = len(pvals) #total 
    order = np.argsort(pvals) #p-values de menor a mayor
    ranked = pvals[order] #aplica ese orden

    qvals = ranked * n / (np.arange(n) + 1) #aplica la formula de correccion FDR
    qvals = np.minimum.accumulate(qvals[::-1])[::-1]  #asegura consistencia en los valores corregidos
    qvals = np.clip(qvals, 0, 1)  # limita los valores al rango valido entre 0 y 1

    out = np.empty(n) #crea un arreglo vacio para almacenar los resultados finales
    out[order] = qvals #regresa los q-values al orden original de los genes
    return out #entrega los valores FDR corregidos

#Carga de datos
print("Cargando datos")

mrna = pd.read_csv(MRNA_PATH)
split_df = pd.read_csv(SPLIT_PATH)
initial_labels = pd.read_csv(INITIAL_LABELS_PATH)

print("mRNA:", mrna.shape)
print("split:", split_df.shape)
print("labels iniciales train:", initial_labels.shape)

#Separa el grupo train
print("\nSeparando Train")

train_ids = split_df.loc[split_df["split"] == "train", "Case_ID"].tolist()

mrna_train = mrna[mrna["Case_ID"].isin(train_ids)].copy()
labels_train = initial_labels[initial_labels["Case_ID"].isin(train_ids)].copy()

mrna_train = mrna_train.sort_values("Case_ID").reset_index(drop=True)
labels_train = labels_train.sort_values("Case_ID").reset_index(drop=True)

assert all(mrna_train["Case_ID"] == labels_train["Case_ID"]), "Los Case_ID no coinciden."

print("Pacientes train:", len(mrna_train))
print(labels_train["label"].value_counts())

#Preprocesamiento (solo train)

print("\n[Preprocesando genes en Train")

X_train = mrna_train.drop(columns=["Case_ID"]).copy()

nan_ratio = X_train.isna().mean()
valid_nan_genes = nan_ratio[nan_ratio < MAX_NAN_RATIO].index
X_train = X_train[valid_nan_genes]

median_train = X_train.median()
X_train = X_train.fillna(median_train)

var_train = X_train.var()
valid_var_genes = var_train[var_train > VAR_THRESHOLD].index
X_train = X_train[valid_var_genes]

mean_train = X_train.mean()
std_train = X_train.std()

X_train_z = (X_train - mean_train) / (std_train + 1e-8)

print("Genes después de filtros:", X_train_z.shape[1])

#Score direccional para basal-clasical usando genes semilla

print("\nCalculando score direccional inicial")

basal_used = [g for g in basal_seed if g in X_train_z.columns] #genes basal disponibles tras el filtrado
classical_used = [g for g in classical_seed if g in X_train_z.columns] #genes classical disponibles tras el filtrado

score_basal_seed = X_train_z[basal_used].mean(axis=1) #score basal inicial por paciente
score_classical_seed = X_train_z[classical_used].mean(axis=1) #score classical inicial por paciente

direction_score = score_basal_seed - score_classical_seed #direccion molecular basal vs clasical
#mustra los genes utilizados
print("Basal seeds usados:", basal_used)
print("Classical seeds usados:", classical_used)

#Correlacion de cada gen con el score direccional
print("\nCalculando correlaciones")

corrs = X_train_z.corrwith(direction_score) #calcula la correlacion de cada gen con la direccion basal-classical

#Busqueda de genes diferencialmente expresados (DEGs) Basal vs Clasical en Train

print("\nCalculando DEGs basal vs classical")

basal_cases = labels_train["label"] == "basal" #identifica pacientes train etiquetados como basal
classical_cases = labels_train["label"] == "classical" #identifica pacientes train etiquetados como clasical

X_basal = X_train_z.loc[basal_cases.values] #matriz de expresion solo para pacientes basal
X_classical = X_train_z.loc[classical_cases.values] #matriz de expresion solo para pacientes clasical

deg_records = [] #lista para guardar resultados por gen

for gene in X_train_z.columns: #bucle para recorrer los genes disponibles
    basal_values = X_basal[gene].values            #Extrae los valores de los pacientes
    classical_values = X_classical[gene].values

    stat, pval = ttest_ind(             #prueba estadistica entre ambos grupos
        basal_values,
        classical_values,
        equal_var=False,
        nan_policy="omit"
    )

    mean_basal = np.mean(basal_values) #expresion promedio del gen en basal
    mean_classical = np.mean(classical_values) #expresion promedio del gen en clasical
    delta = mean_basal - mean_classical #calcula la diferencia de expresion

    deg_records.append({ #guarda los resultados del gen actual
        "Gene": gene,  #nombre del gen
        "mean_basal_z": mean_basal,  #media normalizada en basal
        "mean_classical_z": mean_classical,  #media normalizada en classical
        "delta_basal_minus_classical": delta, #diferencia entre ambos grupos
        "p_value": pval, #significancia estadistica
        "corr_with_basal_direction": corrs[gene] #correlacion con direccion basal-clasical
    })

deg_df = pd.DataFrame(deg_records) #convierte la lista en tabla
deg_df["fdr"] = benjamini_hochberg(deg_df["p_value"].fillna(1).values)  #corrige p-values por multiples comparaciones

#Seleccion de firma expandida

print("\nSeleccionando firma expandida")

basal_candidates = deg_df[  # seleccion de genes candidatos para la firma basal
    (deg_df["corr_with_basal_direction"] >= CORR_THRESHOLD) & #correlacion positiva con direccion basal
    (deg_df["delta_basal_minus_classical"] > 0) &  #mayor expresion en basal que en clasical
    (deg_df["fdr"] <= FDR_THRESHOLD) #diferencia significativa tras correccion FDR
].copy()

classical_candidates = deg_df[ #seleccion de genes candidatos para la firma classical
    (deg_df["corr_with_basal_direction"] <= -CORR_THRESHOLD) & #correlacion negativa con direccion basal
    (deg_df["delta_basal_minus_classical"] < 0) & #mayor expresion en classical que en basal
    (deg_df["fdr"] <= FDR_THRESHOLD)  #diferencia significativa tras correccion FDR
].copy()

basal_candidates = basal_candidates.sort_values( #ordena candidatos basal
    ["fdr", "corr_with_basal_direction", "delta_basal_minus_classical"], # criterios de orden
    ascending=[True, False, False] #menor FDR, mayor correlacion y mayor diferencia basal
)

classical_candidates = classical_candidates.sort_values( #ordena candidatos classical
    ["fdr", "corr_with_basal_direction", "delta_basal_minus_classical"], #criterios de orden
    ascending=[True, True, True]  #menor FDR, correlacion mas negativa y mayor diferencia hacia classical
)

basal_expanded = list(dict.fromkeys( #firma basal expandida sin genes repetidos
    basal_used + basal_candidates["Gene"].head(TOP_N_PER_CLASS).tolist() #semillas basal + mejores candidatos
))

classical_expanded = list(dict.fromkeys( #genera firma classical expandida sin genes repetidos
    classical_used + classical_candidates["Gene"].head(TOP_N_PER_CLASS).tolist() #semillas classical + mejores candidatos
))

print("Genes candidatos basal:", len(basal_candidates)) #muestra el numero de candidatos basal encontrados
print("Genes candidatos classical:", len(classical_candidates)) # numero de classical encontrados
print("Firma basal final:", len(basal_expanded)) # tamaño final de la firma basal
print("Firma classical final:", len(classical_expanded)) # tamaño final de la firma classical

signature_df = pd.DataFrame({  #crea tabla con los genes de la firma expandida
    "Gene": basal_expanded + classical_expanded, #lista total de genes seleccionados
    "signature": ["basal"] * len(basal_expanded) + ["classical"] * len(classical_expanded), #grupo asignado a cada gen
    "is_seed": [ 
        g in basal_seed for g in basal_expanded #indica si el gen basal proviene de la semilla original
    ] + [       
        g in classical_seed for g in classical_expanded  #indica si el gen classical proviene de la semilla original
    ]
})

#Aplica la firma final a todos los pacientes

print("\nAplicando firma expandida a todos los pacientes")

X_all = mrna[["Case_ID"] + list(valid_var_genes)].copy() #toma case id y los genes filtrados

case_ids_all = X_all["Case_ID"].copy() #guarda los ids de los pacientes
X_all_values = X_all.drop(columns=["Case_ID"]) #elimina case id y queda solo expresion genica

X_all_values = X_all_values.fillna(median_train) #rellena valores faltantes con la mediana calculada en train
X_all_z = (X_all_values - mean_train) / (std_train + 1e-8) #normaliza usando media y desviacion de train

score_basal_expanded = X_all_z[basal_expanded].mean(axis=1)  #calcula score basal expandido
score_classical_expanded = X_all_z[classical_expanded].mean(axis=1) #calcula score classical expandido

labels_expanded = np.where(                           # asigna etiqueta segun el score dominante
    score_basal_expanded > score_classical_expanded,
    "basal",
    "classical"
)

results_all = pd.DataFrame({     #crea tabla con resultados finales
    "Case_ID": case_ids_all,    #identificador del paciente
    "score_basal_expanded": score_basal_expanded,  #score basal final
    "score_classical_expanded": score_classical_expanded, #score clasical final
    "score_delta_basal_minus_classical": score_basal_expanded - score_classical_expanded,  # diferencia
    "label_expanded": labels_expanded  #etiqueta molecular final
})

results_all = results_all.merge(  # agrega informacion de split y slide seleccionada
    split_df[["Case_ID", "split", "selected_slide_id", "selected_slide_n_patches"]], #une las columnas especificas
    on="Case_ID", #se unen gracias a case id en comun
    how="left" #añade a la izquierda
)

print("\nDistribución global:")
print(results_all["label_expanded"].value_counts()) #muestra el conteo global de etiquetas

print("\nDistribución por split:")
print(pd.crosstab(results_all["split"], results_all["label_expanded"])) #tabla cruzada split vs etiqueta


#Guarda
deg_path = OUT_DIR / "mRNA_UNI_MIL_3_DEG_correlation_table.csv"  #ruta de tabla DEG y correlaciones
signature_path = OUT_DIR / "mRNA_UNI_MIL_3_expanded_signature_genes.csv" #ruta de genes de firma expandida
labels_path = OUT_DIR / "mRNA_UNI_MIL_3_all_labels_expanded_signature.csv" #ruta de etiquetas finales

deg_df.to_csv(deg_path, index=False) #guarda tabla de genes evaluados
signature_df.to_csv(signature_path, index=False) #guarda firma expandida final
results_all.to_csv(labels_path, index=False) #guarda etiquetas finales por paciente

print("\nArchivos generados:")
print(deg_path)  # muestra la ruta de tabla DEG
print(signature_path)  #muestra la ruta de firma expandida
print(labels_path) #muestra la ruta de etiquetas finales


4. Generacion de embeddings UNI para la slide seleccionada por paciente

In [ ]:
#Librerias
from __future__ import annotations #compatibilidad con anotaciones modernas
import re #expresiones regulares
from pathlib import Path #para rutas
import numpy as np #arreglos numericos
import pandas as pd #para tablas
from PIL import Image # lectura de imagenes
from tqdm import tqdm #barra de progreso
import torch #manejo de tensores y GPU
import timm # carga de modelos preentrenados
from huggingface_hub import login #acceso a modelos de HuggingFace
from timm.data import resolve_data_config #configuracion de preprocesamiento del modelo
from timm.data.transforms_factory import create_transform #transformaciones para imagenes


#Rutas
BASE1 = Path(r"C:\Users\Usuario\Python\Tesis\outputs_pipeline_wsi_omics\mRNA_UNI_MIL_1")
BASE3 = Path(r"C:\Users\Usuario\Python\Tesis\outputs_pipeline_wsi_omics\mRNA_UNI_MIL_3")

LABELS_PATH = BASE3 / "mRNA_UNI_MIL_3_all_labels_expanded_signature.csv"
PATCH_INDEX_PATH = BASE1 / "mRNA_UNI_MIL_1_patch_index_tumor.csv"

OUT_DIR = Path(r"C:\Users\Usuario\Python\Tesis\outputs_pipeline_wsi_omics\mRNA_UNI_MIL_4")
EMB_DIR = OUT_DIR / "embeddings_uni_selected_slides"

OUT_DIR.mkdir(parents=True, exist_ok=True)
EMB_DIR.mkdir(parents=True, exist_ok=True)

#Parametros
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 16 #numero de parches procesados por lote
print("Dispositivo:", DEVICE)

#Funciones
def extract_xy_from_patch_name(filename: str): #extrae coordenadas x,y desde el nombre del parche
    name = Path(filename).stem #obtiene nombre del archivo sin extension

    mx = re.search(r"_x(\d+)", name, flags=re.IGNORECASE) #busca coordenadas
    my = re.search(r"_y(\d+)", name, flags=re.IGNORECASE)

    x = int(mx.group(1)) if mx else None #convierte los valores a enteros
    y = int(my.group(1)) if my else None

    return x, y #entrega las coordenadas


def load_uni_model(): #carga el modelo UNI y sus transformaciones

    print("\nLogin HuggingFace...") # inicia o valida sesion en HuggingFace
    login()

    print("\nCargando modelo UNI...")

    model = timm.create_model(   #crea el modelo desde timm/HuggingFace
        "hf-hub:MahmoodLab/UNI", #repositorio del modelo UNI
        pretrained=True,   #carga pesos preentrenados
        num_classes=0,  #elimina capa de clasificacion
        init_values=1e-5,  #parametro requerido por el modelo
    )

    model.eval().to(DEVICE) #modo evaluacion

    config = resolve_data_config(model.pretrained_cfg, model=model) #obtiene configuracion de entrada del modelo
    transform = create_transform(**config) #crea transformaciones requeridas por UNI

    return model, transform #devuelve el modelo y preprocesamiento


def process_patient(model, transform, patient_row, patch_index, save_path: Path): #a partir de las slides genera enbeddings UNI
    case_id = str(patient_row["Case_ID"]).strip() #obtiene id del paciente
    slide_id = str(patient_row["selected_slide_id"]).strip() #obtiene slide seleccionada

    patches_df = patch_index[ #filtra parches del paciente y slide seleccionada
        (patch_index["Case_ID"] == case_id) &
        (patch_index["Slide_ID"] == slide_id)
    ].copy()

    if patches_df.empty: #verifica presencia de parches
        print(f"Sin parches para {case_id} / {slide_id}")
        return None #detiene el procesamiento de ese paciente

    patches_df = patches_df.sort_values(["y", "x"]).reset_index(drop=True) #ordena parches espacialmente

    embeddings = [] #lista para guardar embeddings por lote
    xs, ys, patch_names, patch_paths_saved = [], [], [], [] #metadatos de cada parche

    batch = [] #lote temporal de imagenes
    meta = [] #metadatos temporales del lote

    for _, patch_row in tqdm(  #recorre los parches de la slide
        patches_df.iterrows(),
        total=len(patches_df),
        desc=f"{case_id} | {slide_id}",
        leave=False
    ):
        patch_path = Path(patch_row["patch_path"]) #ruta del parche

        if not patch_path.exists(): #verifica si el archivo existe
            print(f"No existe el parche: {patch_path}")
            continue #salta al siguiete parche

        try:
            img = Image.open(patch_path).convert("RGB") #abre la imagen y convierte a formato RGB
            img_tensor = transform(img) #aplica transformaciones de UNI

            x = patch_row["x"]  #obtiene las coordenadas
            y = patch_row["y"]

            if pd.isna(x) or pd.isna(y):                           #si faltan coordenadas utiliza el nombre del archivo para crearlas
                x, y = extract_xy_from_patch_name(patch_path.name)

            batch.append(img_tensor) #agrega imagen procesada al lote
            meta.append((patch_path.name, str(patch_path), int(x), int(y))) #guarda metadatos del parche

            if len(batch) == BATCH_SIZE: #si el lote alcanza el tamano definido
                batch_tensor = torch.stack(batch).to(DEVICE) # convierte lote a tensor y lo envia a GPU

                with torch.no_grad():  #evita calcular gradientes
                    feats = model(batch_tensor)  #genera embeddings UNI

                embeddings.append(feats.cpu().numpy())  #guarda embeddings en CPU como numpy

                for name, path_str, x_, y_ in meta:     #guarda metadatos del lote procesado
                    patch_names.append(name)            #nombre del parche
                    patch_paths_saved.append(path_str)  #ruta del parche
                    xs.append(x_)      #coordenadas
                    ys.append(y_)

                batch, meta = [], []  #reinicia las listas de lote y metadatos

        except Exception as e: #captura errores en parches individuales
            print(f"Error en parche {patch_path.name}: {e}") 

    if batch:   #procesa el ultimo lote si quedo incompleto
        batch_tensor = torch.stack(batch).to(DEVICE) #convierte lote final a tensor

        with torch.no_grad(): #evita calcular gradientes
            feats = model(batch_tensor) #tambien genera embeddings UNI

        embeddings.append(feats.cpu().numpy()) #guarda embeddings del lote final

        for name, path_str, x_, y_ in meta:     #guarda metadatos del lote final
            patch_names.append(name)            #nombre del parche
            patch_paths_saved.append(path_str)  #ruta del parche
            xs.append(x_)       #coordenadas
            ys.append(y_)

    if len(embeddings) == 0:  #verifica si no se genero ningun embedding
        print(f" No se generaron embeddings para {case_id}")
        return None

    embeddings = np.concatenate(embeddings, axis=0).astype(np.float32) #se une todos los embeddings en una matriz

    np.savez_compressed(     #guarda embeddings y metadatos en archivo comprimido
        save_path,
        embeddings=embeddings, #matriz de embeddings UNI
        x=np.array(xs, dtype=np.int32), #coordenadas x
        y=np.array(ys, dtype=np.int32), #coordenadas y
        patch_names=np.array(patch_names), #nombres de parches
        patch_paths=np.array(patch_paths_saved), #rutas de parches
        Case_ID=case_id, #id del paciente
        Slide_ID=slide_id, #id de la slide
        label_expanded=str(patient_row["label_expanded"]), #etiqueta molecular
        split=str(patient_row["split"]),  #split del paciente
    )

    return {   #devuelve resumen del paciente procesado
        "Case_ID": case_id,
        "Slide_ID": slide_id,
        "split": patient_row["split"],
        "label_expanded": patient_row["label_expanded"],
        "n_embeddings": embeddings.shape[0],
        "embedding_dim": embeddings.shape[1],
        "embedding_path": str(save_path),
    }


#Procesamiento principal
print("\nCargando labels y patch index")

labels_df = pd.read_csv(LABELS_PATH) #carga etiquetas moleculares y datos de split
patch_index = pd.read_csv(PATCH_INDEX_PATH) #carga indice completo de parches tumorales

print("Labels:", labels_df.shape)
print("Patch index:", patch_index.shape)

required_cols = ["Case_ID", "selected_slide_id", "split", "label_expanded"] #columnas necesarias en labels
for col in required_cols: #recorre columnas requeridas
    if col not in labels_df.columns: #verifica si la columna existe
        raise ValueError(f"Falta columna en labels_df: {col}") #se detiene el script si falta alguna

for col in ["Case_ID", "Slide_ID", "patch_path", "x", "y"]: # columnas necesarias en patch index
    if col not in patch_index.columns:  #verifica existencia de la columna
        raise ValueError(f"Falta columna en patch_index: {col}") #detiene el script si falta alguna

print("\nCargando UNI")
model, transform = load_uni_model() #carga modelo UNI y transformaciones de imagen

print("\nExtrayendo embeddings")

summary_records = [] #lista para almacenar resumen de procesamiento

for split_name in ["train", "val", "test"]:  #procesa cada grupo por separado
    print(f"\n PROCESANDO {split_name.upper()} ") # muestra split actual

    split_df = labels_df[labels_df["split"] == split_name].copy() #selecciona pacientes del split actual
    split_out_dir = EMB_DIR / split_name  #carpeta de salida para ese split
    split_out_dir.mkdir(parents=True, exist_ok=True) #crea carpeta si no existe

    for _, row in split_df.iterrows(): #recorre cada paciente del split
        case_id = str(row["Case_ID"]).strip() #obtiene id del paciente
        slide_id = str(row["selected_slide_id"]).strip()  #obtiene slide seleccionada

        save_path = split_out_dir / f"{case_id}_{slide_id}_UNI.npz" #nombre del archivo de embeddings

        if save_path.exists(): #verifica si los embeddings ya existen
            print(f"{case_id} | {slide_id}: ya existe") #evita reprocesar

            summary_records.append({                     #agrega registro al resumen
                "Case_ID": case_id,                      #identificador del paciente
                "Slide_ID": slide_id,                    #identificador de la slide
                "split": row["split"],                   #grupo train, val o test
                "label_expanded": row["label_expanded"], #etiqueta molecular
                "n_embeddings": np.nan,                  #no se recalcula
                "embedding_dim": np.nan,                 #no se recalcula
                "embedding_path": str(save_path),        #ruta del archivo
                "status": "exists_skipped"               #estado del procesamiento
            })
            continue     #pasa al siguiente paciente

        result = process_patient( #genera embeddings UNI para el paciente
            model=model, #modelo UNI
            transform=transform, #transformaciones de imagen
            patient_row=row, #informacion del paciente
            patch_index=patch_index, #indice de parches
            save_path=save_path  #ruta de salida
        )

        if result is not None:  #verifica si el procesamiento fue exitoso
            result["status"] = "created" #se marca como creado
            summary_records.append(result) #agrega resultado al resumen
        else:  #si falla el procesamiento
            summary_records.append({                      
                "Case_ID": case_id,                        #id del paciente
                "Slide_ID": slide_id,                      #id de la slide
                "split": row["split"],                     #grupo correspondiente
                "label_expanded": row["label_expanded"],   #etiqueta molecular
                "n_embeddings": 0,                         #no se generaron embeddings
                "embedding_dim": np.nan,                   #dimension desconocida
                "embedding_path": str(save_path),          #ruta 
                "status": "failed"                         #estado de fallo
            })

print("\nGuardando resumen")

summary_df = pd.DataFrame(summary_records) #convierte resumen a DataFrame

summary_path = OUT_DIR / "mRNA_UNI_MIL_4_embeddings_summary.csv" #ruta del resumen
summary_df.to_csv(summary_path, index=False) #guarda resumen en CSV

print("\nProceso terminado")
print("Embeddings guardados en:", EMB_DIR)  #muestra carpeta de embeddings
print("Resumen guardado en:", summary_path) #muestra ubicacion del resumen

print("\nDistribución por estado:")
print(summary_df["status"].value_counts()) #muestra cuantos fueron creados, omitidos o fallidos

print("\nDistribución por split:")
print(summary_df.groupby(["split", "label_expanded"]).size()) #muestra cantidad de pacientes por split y etiqueta molecular

5. Clasificacion molecular mediante Graph Attention Networks (GAT) ponderadas por pureza molecular

In [ ]:
#librerias
import json #lectura y escritura de archivos json
import random #generacion de numeros aleatorios
from pathlib import Path # manejo de rutas y archivos

import numpy as np #operaciones numericas y arreglos
import pandas as pd #manejo de tablas y archivos CSV

import torch #framework principal de aprendizaje profundo
import torch.nn as nn #capas y modulos de redes neuronales
import torch.nn.functional as F #funciones auxiliares para redes neuronales

from sklearn.neighbors import NearestNeighbors #identifica vecinos mas cercanos para construir grafos espaciales
from sklearn.metrics import (
    accuracy_score,             #calcula accuracy
    balanced_accuracy_score,    #calcula balanced accuracy
    f1_score,                   # calcula F1-score
    roc_auc_score,             #calcula AUC de la curva ROC
    confusion_matrix,          # genera matriz de confusion
    classification_report,     # genera reporte completo de clasificacion
)

from torch_geometric.data import Data #estructura utilizada para almacenar grafos
from torch_geometric.nn import GATConv #capa Graph Attention Network (GAT)

#Metadatos y rutas
SEED = 42 #semilla para reproducibilidad
DEVICE = "cuda" if torch.cuda.is_available() else "cpu" #usa GPU si esta disponible, caso contrario CPU

EPOCHS = 80 #numero maximo de epocas de entrenamiento
LR = 1e-4 #tasa de aprendizaje del optimizador
WEIGHT_DECAY = 1e-4 #regularizacion para reducir sobreajuste
PATIENCE = 15 #epocas sin mejora antes de aplicar early stopping

K_NEIGHBORS = 4 #numero de vecinos espaciales usados para construir el grafo
UNI_DIM = 1024 #dimension de los embeddings generados por UNI
HIDDEN_DIM = 128 #dimension interna de las capas ocultas del modelo
DROPOUT = 0.6 #proporcion de neuronas apagadas durante entrenamiento
GAT_HEADS = 4 #numero de cabezas de atencion en GATConv

LABEL_MAP = {"classical": 0, "basal": 1} #convierte etiquetas de texto a numeros
INV_LABEL_MAP = {0: "classical", 1: "basal"} #convierte numeros a etiquetas de texto

BASE4 = Path(r"C:\Users\Usuario\Python\Tesis\outputs_pipeline_wsi_omics\mRNA_UNI_MIL_4")
SUMMARY_PATH = BASE4 / "mRNA_UNI_MIL_4_embeddings_summary.csv"

BASE31 = Path(r"C:\Users\Usuario\Python\Tesis\outputs_pipeline_wsi_omics\mRNA_UNI_MIL_3_1")
PURITY_LABELS_PATH = BASE31 / "mRNA_UNI_MIL_3_1_labels_purity_filtered.csv"

OUT_DIR = Path(r"C:\Users\Usuario\Python\Tesis\outputs_pipeline_wsi_omics\mRNA_UNI_MIL_5_GAT_weighted_purity")
OUT_DIR.mkdir(parents=True, exist_ok=True)

GRAPH_DIR = OUT_DIR / "graphs_selected_slides" #grafos
GRAPH_DIR.mkdir(parents=True, exist_ok=True)

ATTN_DIR = OUT_DIR / "attention_scores"
ATTN_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = OUT_DIR / "mRNA_UNI_MIL_5_GAT_weighted_purity_best_model.pt"
PRED_PATH = OUT_DIR / "mRNA_UNI_MIL_5_GAT_weighted_purity_predictions.csv"
METRICS_PATH = OUT_DIR / "mRNA_UNI_MIL_5_GAT_weighted_purity_metrics.json"
HISTORY_PATH = OUT_DIR / "mRNA_UNI_MIL_5_GAT_weighted_purity_history.csv"

print("DEVICE:", DEVICE)


#Fija semilla aleatoria 

def set_seed(seed):
    random.seed(seed)                  #fija la semilla de la libreria random
    np.random.seed(seed)               #fija la semilla de numpy
    torch.manual_seed(seed)            #fija la semilla de PyTorch en CPU
    torch.cuda.manual_seed_all(seed)   # fija la semilla de PyTorch en la GPU

set_seed(SEED) #aplica la semilla que definimos en la configuracion


#Construye el grafo

    # Construccion de conexiones espaciales entre parches vecinos
def build_knn_edges(x_coords, y_coords, k=4):  #construye conexiones entre parches cercanos
    coords = np.stack([x_coords, y_coords], axis=1).astype(np.float32) #une coordenadas x,y en una matriz
    n = coords.shape[0] #numero total de parches o nodos

    if n <= 1:  #si hay uno o ningun parche, no se pueden crear conexiones
        return torch.empty((2, 0), dtype=torch.long) #devuelve grafo sin aristas

    k_eff = min(k + 1, n) #ajusta numero de vecinos para no superar el total de nodos

    nbrs = NearestNeighbors(n_neighbors=k_eff, algorithm="auto") #crea buscador de vecinos cercanos
    nbrs.fit(coords) #ajusta el buscador usando las coordenadas espaciales

    _, indices = nbrs.kneighbors(coords) #obtiene indices de los vecinos mas cercanos

    edges = [] #lista para guardar conexiones entre nodos

    for i in range(n): #bucle que recorre cada parche como nodo origen
        for j in indices[i]: #recorre los vecinos cercanos del nodo actual
            if i == j: #evita conectar un nodo consigo mismo
                continue # salta esa conexion

            edges.append([i, j]) #agrega conexion de i hacia j
            edges.append([j, i]) #agrega conexion inversa de j hacia i

    edge_index = torch.tensor(edges, dtype=torch.long).T #convierte conexiones al formato requerido por PyTorch Geometric
    edge_index = torch.unique(edge_index, dim=1)         # elimina conexiones repetidas

    return edge_index  #devuelve matriz de aristas del grafo

    # Construccion del grafo histologico a partir de embeddings y coordenadas espaciales
def create_graph_from_npz(row): #convierte una slide en un grafo para GAT
    case_id = str(row["Case_ID"]) #identificador del paciente
    slide_id = str(row["Slide_ID"])  #identificador de la slide
    label_str = str(row["label_expanded"]) #etiqueta molecular basal o clasical
    split = str(row["split"]) #grupo train, val o test

    npz_path = Path(row["embedding_path"]) #ruta del archivo de embeddings UNI
    data_npz = np.load(npz_path, allow_pickle=True) #carga embeddings y metadatos del archivo npz

    embeddings = data_npz["embeddings"].astype(np.float32) #embeddings UNI de los parches
    x_coords = data_npz["x"].astype(np.float32)  #coordenadas de los parches
    y_coords = data_npz["y"].astype(np.float32)
    patch_names = data_npz["patch_names"].astype(str) #nombres de los parches

    edge_index = build_knn_edges(x_coords, y_coords, k=K_NEIGHBORS) # genera conexiones espaciales entre parches
 
    data = Data( # crea objeto grafo de PyTorch geometric
        x=torch.tensor(embeddings, dtype=torch.float32),  # nodos del grafo representados por embeddings UNI
        edge_index=edge_index, # conexiones espaciales entre nodos
        y=torch.tensor([LABEL_MAP[label_str]], dtype=torch.long), # etiqueta numerica del paciente
    )

    data.case_id = case_id    # almacena identificador del paciente
    data.slide_id = slide_id   # almacena identificador de la slide
    data.split = split         # almacena grupo train, val o test
    data.label_str = label_str  # almacena etiqueta molecular original
    data.x_coord = torch.tensor(x_coords, dtype=torch.float32)   # almacena coordenadas x
    data.y_coord = torch.tensor(y_coords, dtype=torch.float32)   # almacena coordenadas y
    data.patch_names = patch_names.tolist() # almacena nombres de los parches

    # Peso de muestra para entrenamiento ponderado
    data.sample_weight = torch.tensor([float(row["sample_weight"])], dtype=torch.float32) #peso basado en la pureza molecular
    data.molecular_confidence = float(row["molecular_confidence"])        #confianza molecular original
    data.molecular_confidence_norm = float(row["molecular_confidence_norm"])   #confianza molecular normalizada

    return data  #devuelve el grafo completo de la slide


#Modelo GAT
class GNNGraphAttention(nn.Module): #define la arquitectura del modelo
    def __init__(self, input_dim=1024, hidden_dim=128, dropout=0.5, heads=4):  #parametros del modelo
        super().__init__() #establece la clase base de PyTorch

        self.dropout = dropout #guarda valor de dropout
        self.heads = heads #guarda numero de cabezas de atencion

        self.proj = nn.Sequential( #bloque de proyeccion inicial
            nn.Linear(input_dim, hidden_dim), #reduce embedding UNI de 1024 a hidden_dim
            nn.ReLU(), #activacion no lineal
            nn.Dropout(dropout), #regularizacion para reducir sobreajuste
        )

        self.gnn1 = GATConv(                   #primera capa GAT
            in_channels=hidden_dim,            #dimension de entrada
            out_channels=hidden_dim // heads,  #dimension por cabeza de atencion
            heads=heads,                       #numero de cabezas de atencion
            concat=True,                       #concatena salidas de las cabezas
            dropout=dropout,                   #dropout dentro de la capa GAT
        )

        self.gnn2 = GATConv(                    #segunda capa GAT
            in_channels=hidden_dim,             #dimension de entrada
            out_channels=hidden_dim // heads,   #dimension por cabeza de atencion
            heads=heads,                        #numero de cabezas de atencion
            concat=True,                        #concatena salidas de las cabezas
            dropout=dropout,                    # dropout dentro de la capa GAT
        )

        self.attention = nn.Sequential(                #bloque de atencion para seleccionar parches relevantes
            nn.Linear(hidden_dim, hidden_dim // 2),    #reduce dimension de representacion
            nn.Tanh(),                                 #activacion usada para calcular atencion
            nn.Linear(hidden_dim // 2, 1),             #genera un score de atencion por parche
        )

        self.classifier = nn.Sequential(               #clasificador final a nivel de slide/paciente
            nn.Linear(hidden_dim, hidden_dim // 2),    #primera capa del clasificador
            nn.ReLU(),                                 #activacion no lineal
            nn.Dropout(dropout),                       #regularizacion
            nn.Linear(hidden_dim // 2, 2),             #salida final para dos clases
        )

    def forward(self, data):                                       # define como los datos pasan por el modelo
        x = data.x                                                 # embeddings UNI de los parches
        edge_index = data.edge_index                               #conexiones espaciales del grafo

        h = self.proj(x)                                           #proyecta embeddings UNI a una dimension menor

        h = self.gnn1(h, edge_index)                               #primera propagacion de informacion entre parches vecinos
        h = F.elu(h)                                               #activacion no lineal
        h = F.dropout(h, p=self.dropout, training=self.training)   #aplica dropout durante entrenamiento

        h = self.gnn2(h, edge_index)                               #segunda propagacion de informacion en el grafo
        h = F.elu(h)                                               # activacion no lineal

        attn_logits = self.attention(h).squeeze(1)                 # calcula score de atencion para cada parche
        attn = torch.softmax(attn_logits, dim=0)                   # convierte scores en pesos de atencion

        graph_emb = torch.sum(h * attn.unsqueeze(1), dim=0)        # combina parches usando atencion

        logits = self.classifier(graph_emb)                        # predice basal o classical

        return logits, attn # devuelve prediccion y atencion por parche


#Metricas
def compute_metrics(y_true, y_pred, y_prob): #calcula metricas principales del modelo
    out = {
        "accuracy": float(accuracy_score(y_true, y_pred)), #proporcion total de predicciones correctas
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)), #accuracy balanceada entre clases
        "f1_basal": float(f1_score(y_true, y_pred, pos_label=1)), #F1-score tomando basal como clase positiva
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(), #matriz de confusion
    }

    try: #calcula AUC
        out["auc"] = float(roc_auc_score(y_true, y_prob)) #calcula AUC usando probabilidad basal
    except Exception: #si no se puede calcular AUC
        out["auc"] = None #deja AUC como valor nulo

    return out  #devuelve metricas calculadas

   #Funcion para evaluar el modelo en train, val o test
def evaluate(model, graphs, criterion, save_attention=False): #evalua el modelo sobre una lista de grafos
    model.eval() #modelo en modo evaluacion

    losses = [] #lista para guardar perdidas
    y_true, y_pred, y_prob = [], [], [] #listas para etiquetas reales, predichas y probabilidades
    records = [] #lista para guardar predicciones por paciente

    with torch.no_grad(): #desactiva calculo de gradientes
        for data in graphs: #recorre cada grafo o paciente
            data = data.to(DEVICE) #envia el grafo a CPU o GPU

            logits, attn = model(data) # obtiene prediccion y atencion del modelo

            # criterion usa reduction="none", por eso hacemos mean(). para obtener estructura PyTorch esperada
            loss = criterion(logits.unsqueeze(0), data.y).mean() #calcula perdida del paciente

            prob_basal = torch.softmax(logits, dim=0)[1].item()  #probabilidad asignada a la clase basal
            pred = int(torch.argmax(logits).item()) #clase predicha por el modelo
            true = int(data.y.item()) #clase real del paciente

            losses.append(loss.item()) #guarda perdida
            y_true.append(true) #guarda etiqueta real
            y_pred.append(pred)  #guarda etiqueta predicha
            y_prob.append(prob_basal) #guarda probabilidad basal

            records.append({                #guarda resumen de prediccion del paciente
                "Case_ID": data.case_id,
                "Slide_ID": data.slide_id,
                "split": data.split,
                "label_true": data.label_str,
                "label_pred": INV_LABEL_MAP[pred],
                "prob_basal": prob_basal,
                "correct": int(pred == true),
                "n_nodes": data.x.shape[0],
                "n_edges": data.edge_index.shape[1],
                "sample_weight": float(data.sample_weight.item()),
                "molecular_confidence": float(data.molecular_confidence),
                "molecular_confidence_norm": float(data.molecular_confidence_norm),
            })

            if save_attention: # si guarda la atencion por parche
                attn_df = pd.DataFrame({ # crea tabla con atencion de cada parche
                    "Case_ID": data.case_id,
                    "Slide_ID": data.slide_id,
                    "patch_name": data.patch_names,
                    "x": data.x_coord.detach().cpu().numpy(),
                    "y": data.y_coord.detach().cpu().numpy(),
                    "attention": attn.detach().cpu().numpy(),
                    "label_true": data.label_str,
                    "label_pred": INV_LABEL_MAP[pred],
                    "prob_basal": prob_basal,
                })

                attn_path = ATTN_DIR / f"{data.case_id}_{data.slide_id}_attention.csv" #ruta del archivo de atencion
                attn_df.to_csv(attn_path, index=False) #guarda atencion por parche

    metrics = compute_metrics(y_true, y_pred, y_prob) #calcula metricas globales del conjunto evaluado
    metrics["loss"] = float(np.mean(losses)) #agrega perdida promedio

    return metrics, pd.DataFrame(records) #devuelve metricas y tabla de predicciones



#Carga de resumen de embeddings y pureza molecular
print("\nCargando resumen de embeddings y pureza molecular")

summary_df = pd.read_csv(SUMMARY_PATH) #carga resumen de embeddings generado en la parte 4
summary_df = summary_df[summary_df["status"].isin(["created", "exists_skipped"])].copy() #conserva pacientes con embeddings disponibles

purity_df = pd.read_csv(PURITY_LABELS_PATH) #carga etiquetas moleculares y pureza

purity_keep = purity_df[ #selecciona solo columnas necesarias de pureza
    ["Case_ID", "molecular_confidence", "molecular_purity_group"]
].copy()

summary_df = summary_df.merge( #une resumen de embeddings con informacion de pureza
    purity_keep,
    on="Case_ID", #cruza ambas tablas por paciente
    how="left" #conserva todos los pacientes del resumen
)

if summary_df["molecular_confidence"].isna().any(): #verifica si algun paciente no tiene pureza molecular
    missing = summary_df[summary_df["molecular_confidence"].isna()]["Case_ID"].tolist()
    raise ValueError(f"Pacientes sin pureza molecular: {missing[:10]}")

# Normalizacion de pureza usando solo conjunto Train
train_conf = summary_df.loc[summary_df["split"] == "train", "molecular_confidence"] #pureza molecular solo de train

conf_min = train_conf.min() #valor minimo de pureza en train
conf_max = train_conf.max() #valor maximo de pureza en train

summary_df["molecular_confidence_norm"] = ( #normaliza pureza molecular entre 0 y 1
    (summary_df["molecular_confidence"] - conf_min) / (conf_max - conf_min + 1e-8)
).clip(0, 1) # limita valores al rango 0-1

# Asignacion de pesos de muestra segun pureza molecular (0 menor y 1 maximo)
summary_df["sample_weight"] = 0.5 + 0.5 * summary_df["molecular_confidence_norm"] #asigna mayor peso a casos mas puros

# Val/test no se usan para entrenamiento, pero quedan con peso neutro
summary_df.loc[summary_df["split"].isin(["val", "test"]), "sample_weight"] = 1.0  #val y test quedan con peso neutro

print("Total pacientes:", len(summary_df)) #muestra numero total de pacientes

print("\nDistribución:")
print(pd.crosstab(summary_df["split"], summary_df["label_expanded"])) #distribucion de clases por split

print("\nPesos TRAIN:")
print(summary_df[summary_df["split"] == "train"]["sample_weight"].describe()) #resumen de pesos en train

print("\nPureza TRAIN por clase:")
print(
    summary_df[summary_df["split"] == "train"] #selecciona solo train
    .groupby("label_expanded")["sample_weight"] #agrupa por clase molecular
    .describe() #resume pesos por clase
)


#Crear o cargar grafos espaciales

print("\nCreando/cargando grafos espaciales")

graph_records = [] #lista para guardar el resumen de cada grafo

for _, row in summary_df.iterrows(): # bucle que recorre cada paciente del resumen
    case_id = str(row["Case_ID"])  #obtiene id del paciente
    slide_id = str(row["Slide_ID"]) # obtiene id de la slide

    graph_path = GRAPH_DIR / f"{case_id}_{slide_id}_graph.pt" #ruta donde se guardara o cargara el grafo

    if graph_path.exists(): #verifica si el grafo ya fue creado previamente
        data = torch.load(graph_path, weights_only=False) #carga el grafo existente

        # actualiza atributos si ya existía el grafo
        data.sample_weight = torch.tensor([float(row["sample_weight"])], dtype=torch.float32) #actualiza peso de muestra
        data.molecular_confidence = float(row["molecular_confidence"]) #actualiza confianza molecular
        data.molecular_confidence_norm = float(row["molecular_confidence_norm"]) #actualiza confianza molecular normalizada
    else: #si el grafo no existe
        data = create_graph_from_npz(row) #crea el grafo desde embeddings UNI y coordenadas
        torch.save(data, graph_path) #guarda el grafo para reutilizarlo despues

    graph_records.append({ #guarda informacion general del grafo
        "Case_ID": case_id,
        "Slide_ID": slide_id,
        "split": data.split,
        "label_expanded": data.label_str,
        "sample_weight": float(row["sample_weight"]),
        "molecular_confidence": float(row["molecular_confidence"]),
        "molecular_confidence_norm": float(row["molecular_confidence_norm"]),
        "molecular_purity_group": row["molecular_purity_group"],
        "n_nodes": int(data.x.shape[0]),
        "n_edges": int(data.edge_index.shape[1]),
        "graph_path": str(graph_path),
    })

graph_summary = pd.DataFrame(graph_records) #convierte registros en tabla
graph_summary_path = OUT_DIR / "mRNA_UNI_MIL_5_GAT_weighted_purity_graph_summary.csv" #ruta del resumen de grafos
graph_summary.to_csv(graph_summary_path, index=False) #guarda resumen de grafos

print(graph_summary[["n_nodes", "n_edges"]].describe()) # muestra estadisticas de nodos y conexiones

   #Funcion para cargar grafos por split
def load_graphs(split_name): #carga grafos correspondientes a train, val o test
    paths = graph_summary.loc[graph_summary["split"] == split_name, "graph_path"].tolist() #obtiene rutas del split
    graphs = [torch.load(p, weights_only=False) for p in paths] #carga cada grafo 
    return graphs #devuelve lista de grafos


train_graphs = load_graphs("train") #carga grafos de entrenamiento
val_graphs = load_graphs("val") #carga grafos de validacion
test_graphs = load_graphs("test") #carga grafos de prueba

print("Train:", len(train_graphs)) #muestra el numero de grafos en train
print("Val:", len(val_graphs))  #muestra el numero de grafos en val
print("Test:", len(test_graphs))  #muestra el numero de grafos en test


#Pesos de clase
print("\nPesos de clase")

train_labels = np.array([int(g.y.item()) for g in train_graphs]) #extrae etiquetas reales de los grafos de entrenamiento
class_counts = np.bincount(train_labels, minlength=2) #cuenta cuantos pacientes hay en cada clase
class_weights = class_counts.sum() / (2 * class_counts + 1e-8) #calcula pesos inversamente proporcionales a la frecuencia de cada clase

class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE) #convierte los pesos a tensor y los envia a la GPU

print("Class counts:", class_counts) #muestra cantidad de pacientes por clase
print("Class weights:", class_weights) #muestra pesos asignados a cada clase


#Modelo
print("\nPreparando modelo GAT")

model = GNNGraphAttention( #crea instancia del modelo GAT
    input_dim=UNI_DIM, #dimension de entrada de los embeddings UNI
    hidden_dim=HIDDEN_DIM, #dimension interna del modelo
    dropout=DROPOUT, #valor de dropout utilizado durante entrenamiento
    heads=GAT_HEADS, #numero de cabezas de atencion
).to(DEVICE)

# reduction="none" para poder multiplicar por sample_weight y no usar "mean"
criterion = nn.CrossEntropyLoss( #define la funcion de perdida
    weight=class_weights_tensor, #aplica pesos para compensar desbalance de clases
    reduction="none" #devuelve perdida individual para cada muestra
)

optimizer = torch.optim.AdamW( #define optimizador AdamW
    model.parameters(), #actualiza todos los parametros del modelo
    lr=LR, #tasa de aprendizaje
    weight_decay=WEIGHT_DECAY, #regularizacion para reducir sobreajuste
)


#Entrenamiento
print("\nEntrenando GAT con pérdida ponderada por pureza molecular")

best_val_bal_acc = -1 #mejor balanced accuracy de validacion encontrada
best_epoch = -1  #epoca donde se obtuvo el mejor modelo
patience_counter = 0 #contador para early stopping
history = [] #lista para guardar metricas por epoca

for epoch in range(1, EPOCHS + 1):  #bucle que recorre las epocas de entrenamiento
    model.train() #activa modo entrenamiento

    random.shuffle(train_graphs)  #mezcla los grafos de entrenamiento

    losses = [] #guarda perdidas de la epoca
    y_true, y_pred, y_prob = [], [], [] #guarda etiquetas reales, predicciones y probabilidades

    for data in train_graphs: #recorre cada grafo de entrenamiento
        data = data.to(DEVICE) #envia el grafo a la GPU

        optimizer.zero_grad() #reinicia gradientes acumulados

        logits, attn = model(data) #obtiene prediccion y atencion del modelo

        raw_loss = criterion(logits.unsqueeze(0), data.y) #calcula perdida sin promediar
        sample_weight = data.sample_weight.to(DEVICE) #obtiene peso de muestra segun pureza molecular
        loss = (raw_loss * sample_weight).mean() #pondera la perdida por pureza molecular

        loss.backward() #calcula gradientes
        optimizer.step() #actualiza parametros del modelo

        prob_basal = torch.softmax(logits, dim=0)[1].item() #probabilidad asignada a basal
        pred = int(torch.argmax(logits).item()) #clase predicha
        true = int(data.y.item()) #clase real

        losses.append(loss.item()) #guarda perdida
        y_true.append(true) #guarda etiqueta real
        y_pred.append(pred) #guarda etiqueta predicha
        y_prob.append(prob_basal) #guarda probabilidad basal

    train_metrics = compute_metrics(y_true, y_pred, y_prob) #calcula metricas de entrenamiento
    train_metrics["loss"] = float(np.mean(losses)) #agrega perdida promedio de entrenamiento

    val_metrics, _ = evaluate(model, val_graphs, criterion, save_attention=False) #evalua el modelo en validacion

    history.append({         #guarda metricas de la epoca
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_bal_acc": train_metrics["balanced_accuracy"],
        "train_auc": train_metrics["auc"],
        "val_loss": val_metrics["loss"],
        "val_bal_acc": val_metrics["balanced_accuracy"],
        "val_auc": val_metrics["auc"],
        "val_f1_basal": val_metrics["f1_basal"],
    })

    print(
        f"Epoch {epoch:03d} | " #numero de epoca
        f"train loss {train_metrics['loss']:.4f} | " #perdida en train
        f"train bal_acc {train_metrics['balanced_accuracy']:.3f} | " #balanced accuracy en train
        f"val loss {val_metrics['loss']:.4f} | "  #perdida en validacion
        f"val bal_acc {val_metrics['balanced_accuracy']:.3f} | " #balanced accuracy en validacion
        f"val auc {val_metrics['auc']}" #AUC en validacion
    )

    if val_metrics["balanced_accuracy"] > best_val_bal_acc: #verifica si el modelo mejora en validacion
        best_val_bal_acc = val_metrics["balanced_accuracy"] #actualiza mejor balanced accuracy
        best_epoch = epoch #guarda la epoca del mejor modelo
        patience_counter = 0 #reinicia contador de early stopping

        torch.save({ #guarda el mejor modelo encontrado
            "model_state_dict": model.state_dict(), #pesos actuales del modelo
            "best_epoch": best_epoch, #epoca del mejor modelo
            "best_val_bal_acc": best_val_bal_acc,  #mejor balanced accuracy de validacion
            "config": { #configuracion usada para entrenar el modelo
                "model_type": "GATConv", #tipo de arquitectura
                "training": "weighted_by_molecular_purity", #estrategia de entrenamiento
                "sample_weight_formula": "0.5 + 0.5 * normalized_molecular_confidence", #formula de ponderacion
                "uni_dim": UNI_DIM, #dimension de entrada UNI
                "hidden_dim": HIDDEN_DIM, #dimension oculta del modelo
                "dropout": DROPOUT, #dropout utilizado
                "gat_heads": GAT_HEADS, #numero de cabezas GAT
                "k_neighbors": K_NEIGHBORS, #numero de vecinos espaciales
                "label_map": LABEL_MAP, #codificacion de etiquetas
            }
        }, MODEL_PATH)

        print("Mejor modelo guardado")
    else:  #si no mejora en validacion
        patience_counter += 1 #aumenta contador de paciencia

    if patience_counter >= PATIENCE: #verifica si se alcanzo el limite sin mejora
        print(f"\nEarly stopping en epoca {epoch}")
        break #detiene el entrenamiento


#Evaluacion final del mejor modelo

print("\nEvaluando el mejor modelo GAT ponderado por pureza molecular")

checkpoint = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False) #carga el mejor modelo guardado
model.load_state_dict(checkpoint["model_state_dict"]) #carga los pesos del mejor modelo

train_metrics, train_pred = evaluate(model, train_graphs, criterion, save_attention=False) #evalua en train sin guardar atencion
val_metrics, val_pred = evaluate(model, val_graphs, criterion, save_attention=True)  #evalua en validacion y guarda atencion
test_metrics, test_pred = evaluate(model, test_graphs, criterion, save_attention=True) #evalua en test y guarda atencion

pred_all = pd.concat([train_pred, val_pred, test_pred], axis=0) #une predicciones de train, val y test
pred_all.to_csv(PRED_PATH, index=False) #guarda predicciones finales

pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)  #guarda historial de entrenamiento por epoca

metrics = { #crea diccionario con metricas y configuracion final
    "best_epoch": int(best_epoch), #mejor epoca segun validacion
    "best_val_bal_acc": float(best_val_bal_acc),  #mejor balanced accuracy en validacion
    "train": train_metrics, #metricas de train
    "val": val_metrics, #metricas de validacion
    "test": test_metrics, #metricas de test
    "class_counts_train": class_counts.tolist(), #cantidad de pacientes por clase en train
    "class_weights": class_weights.tolist(), #pesos de clase utilizados
    "sample_weight_formula": "0.5 + 0.5 * normalized_molecular_confidence", #formula de peso por pureza molecular
    "config": { #configuracion del modelo
        "model_type": "GATConv", #tipo de modelo
        "training": "weighted_by_molecular_purity", #estrategia de entrenamiento
        "k_neighbors": K_NEIGHBORS, #numero de vecinos espaciales
        "gat_heads": GAT_HEADS, #numero de cabezas de atencion
        "uni_dim": UNI_DIM, #dimension de embeddings UNI
        "hidden_dim": HIDDEN_DIM, #dimension oculta
        "dropout": DROPOUT, #dropout usado
        "lr": LR, #tasa de aprendizaje
        "weight_decay": WEIGHT_DECAY, #regularizacion
        "epochs": EPOCHS, #numero maximo de epocas
        "patience": PATIENCE, #paciencia para early stopping
    }
}

with open(METRICS_PATH, "w", encoding="utf-8") as f:  #abre archivo JSON para guardar metricas
    json.dump(metrics, f, indent=4) #guarda metricas y configuracion en formato JSON

print("\nTRAIN:")
print(train_metrics) #muestra metricas de train

print("\nVAL:")
print(val_metrics) #muestra metricas de validacion

print("\nTEST:")
print(test_metrics) #muestra metricas de test

print("\nClassification report TEST:")
print( #genera reporte detallado de clasificacion en test
    classification_report(
        test_pred["label_true"], #etiquetas reales
        test_pred["label_pred"], #etiquetas predichas
        labels=["classical", "basal"], #orden de clases en el reporte
    )
)

print("\nArchivos generados:")
print("Grafo resumen:", graph_summary_path) #ruta del resumen de grafos
print("Modelo:", MODEL_PATH) #ruta del mejor modelo
print("Predicciones:", PRED_PATH) #ruta de predicciones
print("Métricas:", METRICS_PATH) #ruta de metricas
print("Historial:", HISTORY_PATH) #ruta del historial de entrenamiento
print("Attention:", ATTN_DIR) #ruta de archivos de atencion

6. Mapas de calor y Top parches

In [ ]:
#Librerias

from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image, ImageDraw #lectura y edicion basica de imagenes
import matplotlib.pyplot as plt  #generacion de graficas

#Rutas
BASE5 = Path(r"C:\Users\Usuario\Python\Tesis\outputs_pipeline_wsi_omics\mRNA_UNI_MIL_5_GAT_weighted_purity")

PRED_PATH = BASE5 / "mRNA_UNI_MIL_5_GAT_weighted_purity_predictions.csv"
ATTN_DIR = BASE5 / "attention_scores"

PATCH_ROOT = Path(r"D:\ProyectoPDAC\Parches_PDAC_Macenko\tumor")

OUT_DIR = Path(r"C:\Users\Usuario\Python\Tesis\outputs_pipeline_wsi_omics\mRNA_UNI_MIL_6_GAT_weighted_purity")

HEATMAP_DIR = OUT_DIR / "test_heatmaps" #carpeta para guardar heatmaps
TOP_PATCH_DIR = OUT_DIR / "test_top_patches" #carpeta para guardar paneles de top patches

OUT_DIR.mkdir(parents=True, exist_ok=True) #crea carpeta principal si no existe
HEATMAP_DIR.mkdir(parents=True, exist_ok=True) #crea carpeta de heatmaps si no existe
TOP_PATCH_DIR.mkdir(parents=True, exist_ok=True) #crea carpeta de top patches si no existe

#Configuraciones
TILE_SIZE = 64 #tamano base de cada parche en el panel
TOP_K = 12  #numero de parches con mayor atencion
CMAP = "hot" #mapa de color usado para los heatmaps

#Funciones
def normalize_array(a): #normaliza un arreglo entre 0 y 1
    a = np.asarray(a, dtype=float) #convierte entrada a arreglo numerico
    if a.max() - a.min() < 1e-12: #evita division por cero si todos los valores son iguales
        return np.zeros_like(a) #devuelve arreglo de ceros
    return (a - a.min()) / (a.max() - a.min()) #normalizacion min-max


def make_attention_heatmap(attn_df, out_path, title):  #genera heatmap de atencion
    df = attn_df.copy() #copia tabla de atenciones
    df["attention_norm"] = normalize_array(df["attention"].values) #normaliza atencion entre 0 y 1

    x_unique = sorted(df["x"].unique()) #obtiene coordenadas x unicas
    y_unique = sorted(df["y"].unique()) #obtiene coordenadas y unicas

    x_to_i = {x: i for i, x in enumerate(x_unique)} #asigna indice de matriz a cada coordenada x
    y_to_i = {y: i for i, y in enumerate(y_unique)} #asigna indice de matriz a cada coordenada y

    W = len(x_unique) #ancho del heatmap
    H = len(y_unique) #alto del heatmap

    heat = np.zeros((H, W), dtype=float) #matriz vacia para el heatmap

    for _, row in df.iterrows(): #recorre cada parche
        xi = x_to_i[row["x"]] #indice x en la matriz
        yi = y_to_i[row["y"]] #indice y en la matriz
        heat[yi, xi] = row["attention_norm"] #asigna atencion normalizada al punto espacial

    plt.figure(figsize=(10, 10)) #crea figura
    plt.imshow(heat, cmap=CMAP) #muestra matriz como heatmap
    plt.colorbar(label="Attention normalizada") #agrega barra de color
    plt.title(title) #agrega titulo
    plt.axis("off") #oculta ejes
    plt.tight_layout() #ajusta margenes
    plt.savefig(out_path, dpi=300) #guarda imagen
    plt.close() #cierra figura


def find_patch_path(patch_name): #busca la ruta real de un parche
    found = list(PATCH_ROOT.rglob(patch_name)) #busca el parche dentro de la carpeta raiz
    if len(found) == 0: #si no encuentra el archivo
        return None #regresa None
    return found[0] #regresa lo primero encontrado


def make_top_patch_panel(attn_df, out_path, title, top_k=12): #crea panel con parches de mayor atencion
    df = attn_df.sort_values("attention", ascending=False).head(top_k).copy() #selecciona top parches

    images = [] #lista para guardar imagenes cargadas

    for _, row in df.iterrows(): #recorre top parches
        patch_name = row["patch_name"] #nombre del parche
        patch_path = find_patch_path(patch_name) #busca ruta del parche

        if patch_path is None: #si no encuentra el parche
            print(f"No se encontro parche: {patch_name}")
            continue #salta al siguiente parche

        img = Image.open(patch_path).convert("RGB").resize((TILE_SIZE * 2, TILE_SIZE * 2)) #carga y redimensiona imagen
        draw = ImageDraw.Draw(img) #permite dibujar sobre la imagen

        txt = f"{row['attention']:.4f}" #texto con valor de atencion
        draw.rectangle([0, 0, 90, 20], fill=(255, 255, 255)) #fondo blanco para texto
        draw.text((4, 3), txt, fill=(0, 0, 0))  #escribe valor de atencion

        images.append(img) #agrega imagen al panel

    if len(images) == 0: #si no se cargo ninguna imagen
        print(f"No se pudieron cargar top patches para {title}")
        return #termina la funcion

    cols = 4 #numero de columnas del panel
    rows = int(np.ceil(len(images) / cols)) #numero de filas necesarias

    panel_w = cols * TILE_SIZE * 2 #ancho del panel
    panel_h = rows * TILE_SIZE * 2 + 70 #alto del panel con espacio para titulo

    panel = Image.new("RGB", (panel_w, panel_h), "white")  #crea imagen blanca para el panel
    draw = ImageDraw.Draw(panel) #permite dibujar sobre el panel

    draw.text((10, 10), title, fill=(0, 0, 0)) #agrega titulo al panel

    y_offset = 60 #espacio superior antes de colocar parches

    for i, img in enumerate(images): #recorre imagenes seleccionadas
        x = (i % cols) * TILE_SIZE * 2 #posicion x del parche en el panel
        y = y_offset + (i // cols) * TILE_SIZE * 2 #posicion y del parche en el panel
        panel.paste(img, (x, y)) #pega parche en el panel

    panel.save(out_path) #guarda panel final

#Ejecucion principal
print("\nCargando predicciones")

pred = pd.read_csv(PRED_PATH) #carga las predicciones del Script 5

test_pred = pred[pred["split"] == "test"].copy() #filtra solo pacientes test

print("Pacientes TEST:", len(test_pred)) #muestra numero de pacientes test
print(test_pred[["Case_ID", "Slide_ID", "label_true", "label_pred", "prob_basal", "correct"]]) #muestra resumen de predicciones


print("\nGenerando heatmaps y top patches")

summary_records = [] #lista para guardar resumen visual

for _, row in test_pred.iterrows(): #recorre pacientes test
    case_id = row["Case_ID"] #id del paciente
    slide_id = row["Slide_ID"] #id de la slide

    attn_path = ATTN_DIR / f"{case_id}_{slide_id}_attention.csv" #ruta del archivo de atencion

    if not attn_path.exists(): #verifica si existe el archivo de atencion
        print(f"No existe attention csv: {attn_path}")
        continue #salta al siguiente paciente

    attn_df = pd.read_csv(attn_path) #carga atenciones por parche

    label_true = row["label_true"] #etiqueta real
    label_pred = row["label_pred"] #etiqueta predicha
    prob_basal = row["prob_basal"] #probabilidad de basal
    correct = int(row["correct"]) #indica si la prediccion fue correcta

    status = "CORRECT" if correct == 1 else "ERROR" #define estado de prediccion

    title = ( #titulo para heatmap y panel
        f"{case_id} | {slide_id}\n"
        f"True: {label_true} | Pred: {label_pred} | " 
        f"P(basal): {prob_basal:.3f} | {status}"
    )

    base_name = f"{case_id}_{slide_id}_{status}_true-{label_true}_pred-{label_pred}" #nombre base de archivos

    heatmap_path = HEATMAP_DIR / f"{base_name}_heatmap.png" #ruta del heatmap
    top_patch_path = TOP_PATCH_DIR / f"{base_name}_top{TOP_K}.png" #ruta del panel de top patches

    make_attention_heatmap(attn_df, heatmap_path, title) #genera heatmap de atencion
    make_top_patch_panel(attn_df, top_patch_path, title, top_k=TOP_K) #genera panel de top parches

    top_df = attn_df.sort_values("attention", ascending=False).head(TOP_K).copy() #tabla con top parches

    top_csv_path = TOP_PATCH_DIR / f"{base_name}_top{TOP_K}.csv" #ruta CSV de top parches
    top_df.to_csv(top_csv_path, index=False) #guarda tabla de top parches

    summary_records.append({ #guarda rutas y datos del paciente
        "Case_ID": case_id,
        "Slide_ID": slide_id,
        "label_true": label_true,
        "label_pred": label_pred,
        "prob_basal": prob_basal,
        "correct": correct,
        "heatmap_path": str(heatmap_path),
        "top_patch_panel": str(top_patch_path),
        "top_patch_csv": str(top_csv_path),
    })


print("\nGuardando resumen")

summary_df = pd.DataFrame(summary_records) #convierte resumen a tabla

summary_path = OUT_DIR / "mRNA_UNI_MIL_6_GAT_weighted_purity_test_visual_summary.csv" #ruta del resumen visual
summary_df.to_csv(summary_path, index=False)  #guarda resumen visual


print("Resumen:", summary_path) #ruta resumen
print("Heatmaps:", HEATMAP_DIR) #ruta heatmaps
print("Top patches:", TOP_PATCH_DIR) #ruta top patches

print("\nDistribución:")
print(summary_df.groupby(["label_true", "label_pred"]).size()) #distribucion de aciertos y errores por clase